In [86]:
# -------------------------------------------------
# Cell 1 – Import, configurazione e inizializzazione client
# -------------------------------------------------
import os, io, threading, requests
import pyaudio, wave
from huggingface_hub import InferenceClient

# ---------- Audio parameters ----------
FORMAT = pyaudio.paInt16          # 16‑bit PCM
CHANNELS = 1                      # mono mic
RATE = 16000                      # 16 kHz (common for Whisper)
CHUNK = 1024                      # buffer size

In [87]:
# -------------------------------------------------
# Cell 2 – Stato globale, thread di registrazione e funzioni di controllo
# -------------------------------------------------
audio_state = {
    "is_recording": False,
    "frames": [],
    "stream": None,
    "p": pyaudio.PyAudio(),
    "thread": None
}

def _recording_thread_logic():
    """Thread che legge dal microfono e salva i campioni."""
    try:
        audio_state["stream"] = audio_state["p"].open(
            format=FORMAT,
            channels=CHANNELS,
            rate=RATE,
            input=True,
            frames_per_buffer=CHUNK,
        )
        print("🎤 Microfono aperto e pronto a registrare.")
    except Exception as e:
        print(f"❌ Impossibile aprire lo stream audio: {e}")
        print("Verifica che il microfono sia collegato e funzionante.")
        audio_state["is_recording"] = False
        return

    audio_state["frames"] = []
    frame_count = 0
    
    while audio_state["is_recording"]:
        try:
            data = audio_state["stream"].read(CHUNK, exception_on_overflow=False)
            audio_state["frames"].append(data)
            frame_count += 1
            
            # Mostra progresso ogni 100 frame (circa 6.4 secondi a 16kHz)
            if frame_count % 100 == 0:
                duration = (frame_count * CHUNK) / RATE
                print(f"⏱️ Registrazione in corso... {duration:.1f}s")
                
        except Exception as e:
            print(f"⚠️ Errore durante la lettura: {e}")
            break

    # Chiusura pulita dello stream
    if audio_state["stream"]:
        audio_state["stream"].stop_stream()
        audio_state["stream"].close()
        audio_state["stream"] = None
    
    total_duration = (len(audio_state["frames"]) * CHUNK) / RATE
    print(f"✅ Registrazione terminata. Durata totale: {total_duration:.1f}s")

def start_recording():
    """Avvia la registrazione."""
    if audio_state["is_recording"]:
        print("⏩ La registrazione è già in corso.")
        return
    
    audio_state["is_recording"] = True
    audio_state["thread"] = threading.Thread(target=_recording_thread_logic, daemon=True)
    audio_state["thread"].start()
    print("🔴 Registrazione avviata – scrivi 'stop' per fermarla.")

def stop_recording():
    """Ferma la registrazione."""
    if not audio_state["is_recording"]:
        print("⏸️ Nessuna registrazione attiva.")
        return
    
    audio_state["is_recording"] = False
    if audio_state["thread"]:
        audio_state["thread"].join()
    audio_state["thread"] = None

print("✅ Funzioni di registrazione definite.")

✅ Funzioni di registrazione definite.


In [88]:
# -------------------------------------------------
# Cell 3 – Funzione di trascrizione con sistema di fallback
# -------------------------------------------------
def transcribe_with_fallback():
    """
    Trascrive l'audio provando automaticamente provider diversi.
    """
    if not audio_state["frames"]:
        print("⚠️ Nessun audio da trascrivere.")
        return

    # Creazione del buffer WAV in memoria
    print("⏳ Preparazione del buffer audio...")
    wav_buf = io.BytesIO()
    with wave.open(wav_buf, "wb") as wf:
        wf.setnchannels(CHANNELS)
        wf.setsampwidth(audio_state["p"].get_sample_size(FORMAT))
        wf.setframerate(RATE)
        wf.writeframes(b''.join(audio_state["frames"]))
    wav_buf.seek(0)

    # Informazioni di debug sul buffer
    audio_data = wav_buf.getvalue()
    print(f"🔍 Dimensione buffer: {len(audio_data):,} bytes")
    print(f"🔍 Header WAV: {audio_data[:12]}")
    
    success = False

    try:
        # Crea un client
        test_client = InferenceClient(
            model="openai/whisper-large-v3",
            provider="fal-ai",
            token=os.getenv("HF_TOKEN_READ", None)
        )
        # Tenta la trascrizione
        if test_client is None:
            raise RuntimeError("❌ Test Client non inizializzato.")
        else: 
            print(f"⏳ Chiamata API in corso ...")
            result = test_client.automatic_speech_recognition(
                audio=wav_buf.getvalue()
            )
            testo = result.get("text", "")
            if testo:
                print(f"\n--- TRASCRIZIONE con {test_client.model}) ---")
                print(testo)
                print("--- FINE ---")
                success = True
            else:
                print(f"⚠️ Il Provider ha restituito testo vuoto")              
                            
    except Exception as e:
        error_msg = str(e)
        print(f"❌ Tentativo col Provider fallito: {error_msg}")
                
    if not success:
        print("\n❌ IMPOSSIBILE TRASCRIVERE L'AUDIO")
        print("💡 Possibili cause:")
        print("   - Il modello non è disponibile via API")
        print("   - Problemi di autenticazione")
        print("   - Problemi di quota/limiti")
        print("   - Formato audio non supportato")
            
    # Pulizia del buffer
    audio_state["frames"] = []
    return success

print("✅ Funzione di trascrizione definita.")

✅ Funzione di trascrizione definita.


In [89]:
# -------------------------------------------------
# Cell 4 – Loop di controllo interattivo
# -------------------------------------------------
def main():
    """Loop principale per il controllo della registrazione e trascrizione."""
    print("\n🎤 SISTEMA DI TRASCRIZIONE AUDIO")
    print("=" * 50)
    print("Comandi disponibili:")
    print("  start   → avvia la registrazione")
    print("  stop    → ferma la registrazione e trascrive")
    print("  exit    → esce dal programma")
    print("=" * 50)
    
    while True:
        try:
            cmd = input("\n🔹 Comando (start/stop/exit): ").strip().lower()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Uscita dal programma.")
            break
        
        if cmd == "start":
            start_recording()
            
        elif cmd == "stop":
            print("⏹️ Fermando la registrazione...")
            stop_recording()
            print("🎯 Iniziando la trascrizione...")
            transcribe_with_fallback()                          
        
        elif cmd == "exit":
            if audio_state["is_recording"]:
                print("⚠️ Registrazione ancora attiva. Ferma prima di uscire.")
                continue
            print("👋 Uscita dal programma.")
            break
            
        else:
            print("❓ Comando non riconosciuto. Usa: start, stop, exit")

print("✅ Loop di controllo definito.")

✅ Loop di controllo definito.


In [90]:
# -------------------------------------------------
# Cell 5 – Avvio del sistema di trascrizione
# -------------------------------------------------

# Avvia il loop principale
main()


🎤 SISTEMA DI TRASCRIZIONE AUDIO
Comandi disponibili:
  start   → avvia la registrazione
  stop    → ferma la registrazione e trascrive
  exit    → esce dal programma



🔹 Comando (start/stop/exit):  start


🔴 Registrazione avviata – scrivi 'stop' per fermarla.
🎤 Microfono aperto e pronto a registrare.
⏱️ Registrazione in corso... 6.4s



🔹 Comando (start/stop/exit):  stop


⏹️ Fermando la registrazione...
✅ Registrazione terminata. Durata totale: 7.7s
🎯 Iniziando la trascrizione...
⏳ Preparazione del buffer audio...
🔍 Dimensione buffer: 245,804 bytes
🔍 Header WAV: b'RIFF$\xc0\x03\x00WAVE'
⏳ Chiamata API in corso ...
❌ Tentativo col Provider fallito: Model openai/whisper-medium is not supported by provider fal-ai.

❌ IMPOSSIBILE TRASCRIVERE L'AUDIO
💡 Possibili cause:
   - Il modello non è disponibile via API
   - Problemi di autenticazione
   - Problemi di quota/limiti
   - Formato audio non supportato



🔹 Comando (start/stop/exit):  exit


👋 Uscita dal programma.
